# Prompt Versioning& A/B Testing

**Module 14 · Lesson 14.2**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- The Prompt Registry
- Semantic Versioning For Prompts
- Deterministic Bucketing
- The A/B Test Itself
- Statistical Significance
- Canary Rollout

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Reproducibility - the lesson uses seeded random values throughout

## You would not ship code that way — so why ship a prompt that way?

> 💡 **Info**
>
> You would never ship a code change without a commit, a pull request, and a review. Yet the most common way a prompt changes in production is a PR titled “*tweaked the prompt to fix the JSON*” — a dozen lines of a triple-quoted string edited in place, no version, no eval, straight to everyone. Three days later the support tickets spike, nobody remembers what the prompt said before, and the fix is a panicked `git blame` on a string. The reason this hurts so much is that the **prompt is the highest-leverage artifact in your app**: when you do not train the model, a few dozen lines of prompt drive behaviour across every request, so one changed word can flip an output format, shift the tone for every customer, or move the cost per call. This lesson gives the prompt the discipline it deserves: a **registry** where every version has a permanent address, **semantic versioning** so a version means one thing forever, **deterministic A/B routing** and a **significance test** so you ship on evidence not vibes, a **canary ramp** that catches what the sample missed, and a **rollback path** for when a change goes wrong. Each piece is a runnable model you can read end to end.

### 🎯 What you will be able to do after this lesson

- **Build** the prompt-lifecycle tooling as runnable models — a versioned registry, a SemVer bumper, a deterministic bucketer, an A/B analyzer, a two-proportion significance test, and a canary state machine.

- **Apply** the SemVer bump rules to real prompt changes, and deterministic hashing to route A/B traffic cleanly.

- **Analyze** an A/B result for statistical significance — is the gap real, or is it noise?

- **Evaluate** when a change is safe to ramp, and exactly what a rollback does and does not undo.

> 📦 **Info**
>
> ✅ Before you startThis builds directly on three earlier ideas. In **14.1** every call emitted a trace with the `prompt_version` field — that field is what makes an A/B test measurable here. In **1.8** you met Bayesian A/B testing and why a bigger sample is more convincing — Step 5 is the frequentist companion. And in **12.6** a CI gate ran checks and blocked a merge on failure — the eval gate that decides an A/B winner is the same idea. Which arm actually won, judged by an LLM-as-judge on a golden set, is the next lesson.

## 60-Second Warm-Up

Flip each card and answer before revealing.

> 📚 **Analogy**
>
> Treating the prompt like a versioned asset is **a library catalogue instead of a pile of loose paper**. In a good library, every edition of a book has a permanent call number: you can pull the 2019 edition or the 2024 edition on demand, see exactly what changed between them, and no one scribbles edits into a shelved copy. A prompt registry is that catalogue — every version has a permanent address, every past version is still retrievable, and a published version is never altered in place. A prompt hard-coded in someone’s file is the opposite: a single loose page that anyone can rewrite, with no record of what it used to say. **Where it breaks down:** a library book is read-only once shelved, but a prompt keeps evolving — which is exactly why it needs the catalogue’s version numbers.

> 📦 **Info**
>
> 🚫 Two misconceptions this lesson kills
> - **“A prompt is just data — tweak it in place.”** A published version must mean exactly one thing forever; editing it in place destroys reproducibility and your ability to answer “what did the model actually see yesterday?”
> - **“The candidate is ahead, ship it.”** A lead on a small sample is usually noise. You ship when the difference is statistically significant — or when you have another reason (cost, latency) and accept “no measurable quality change”.

> 💡 **Info**
>
> ⚠️ Anti-patternThe prompt change that reads “*tweaked the prompt to fix the JSON issue*”: a dozen lines of a triple-quoted string edited directly, no version bump, no eval run, shipped to every user at once. It works in the demo and breaks in production, and because nothing recorded the old text, recovery is archaeology. Every step in this lesson exists to make that PR impossible.

---

## 🎯 Concept 1: The Prompt Registry

### The Prompt Registry

The prompt is the model when you do not train, so store it as data with a permanent address: a versioned file (YAML in git), loaded by version, rendered with variables. The file is the contract - prompt, model, parameters, and the eval bar. A published version is never edited in place; a hardcoded string is one mutable state with no history.

Start with where the prompt lives. The wrong answer is a triple-quoted string in `main.py`: it has exactly one state, and the only history is whatever git happened to capture on the source file. The right answer is a **registry** — the prompt stored as *data*, typically a YAML file per version checked into git, with a permanent address of **id plus semantic version**. Code **loads it by version** and renders the variables. The file carries everything the call needs: the template, the model, the parameters, the eval set, and the expected quality bar — **the file is the contract**, and code just loads it. Two rules follow immediately: a **published version is never edited in place** (to change it, you publish a new version), and every call **logs which version produced it** (the `prompt_version` from 14.1). The block builds a tiny registry and loads two versions, keyless. (In production the template uses a real engine like Jinja with strict-undefined so a missing variable crashes early instead of silently rendering empty; the illustrative artifact at the end shows that wiring.)

> 📚 **Analogy**
>
> It is **a library call number**. Every edition of a book sits at a permanent address on the shelf; you ask for the 2019 edition and you get exactly that, unchanged, forever. You do not walk up to a shelved book and rewrite a paragraph in pencil — you publish a new edition. A prompt registry works the same way: `review-classifier` version 1.0.0 is a fixed address that always returns the same text, and to change it you publish 1.1.0. The hardcoded string is the book with no call number that everyone scribbles in.

Your prompt is a triple-quoted string in main.py. What is the core problem when someone asks ‘what did the prompt say last Tuesday?’

**📝 `01_prompt_registry.py`** - *runnable*

In [ ]:
# The prompt IS the model when you do not train, so it needs a REGISTRY: every version has a permanent address
# (id + semver), stored as data (YAML in git), loaded by version, rendered with variables. The file is the contract.
REGISTRY = {   # in a real app these are prompts/<id>-v<version>.yaml files in git; here, one dict for a keyless demo
    ("review-classifier", "1.0.0"): {"model": "claude-sonnet-4-6", "temperature": 0.0, "max_tokens": 100,
        "template": "Classify this review into refund/delivery/food/service/other.\nReply with ONLY the label.\nReview: {review}",
        "eval_set": "golden/review-classifier.jsonl", "expected_pass_rate": 0.92},
    ("review-classifier", "1.1.0"): {"model": "claude-sonnet-4-6", "temperature": 0.0, "max_tokens": 100,
        "template": "Classify this review into refund/delivery/food/service/other.\nExample: 'cold pizza, want money back' -> refund\nReply with ONLY the label.\nReview: {review}",
        "eval_set": "golden/review-classifier.jsonl", "expected_pass_rate": 0.95},
}
def load_prompt(pid, version):
    key = (pid, version)
    if key not in REGISTRY:
        raise KeyError("no such prompt version: {} v{}".format(pid, version))
    return REGISTRY[key]
def render(spec, **kw):
    out = spec["template"]
    for k, v in kw.items():
        out = out.replace("{" + k + "}", v)
    return out
p10 = load_prompt("review-classifier", "1.0.0")
p11 = load_prompt("review-classifier", "1.1.0")
print("Loaded review-classifier v1.0.0: model={}, temp={}, eval bar={}".format(p10["model"], p10["temperature"], p10["expected_pass_rate"]))
print("Loaded review-classifier v1.1.0: model={}, temp={}, eval bar={}".format(p11["model"], p11["temperature"], p11["expected_pass_rate"]))
print("v1.1.0 adds a few-shot example -> its rendered prompt is {} chars vs v1.0.0's {} chars.".format(
      len(render(p11, review="pizza was cold")), len(render(p10, review="pizza was cold"))))
print("Both versions still exist and are loadable by address; a published version is NEVER edited in place.")
print("A hardcoded string in main.py has exactly one mutable state and no history - the registry is the fix.")

# Output:
# Loaded review-classifier v1.0.0: model=claude-sonnet-4-6, temp=0.0, eval bar=0.92
# Loaded review-classifier v1.1.0: model=claude-sonnet-4-6, temp=0.0, eval bar=0.95
# v1.1.0 adds a few-shot example -> its rendered prompt is 160 chars vs v1.0.0's 111 chars.
# Both versions still exist and are loadable by address; a published version is NEVER edited in place.
# A hardcoded string in main.py has exactly one mutable state and no history - the registry is the fix.

- Each version is **loaded by its address** (id plus semantic version) — here v1.0.0 and v1.1.0, each with its own eval bar.
- The file carries the **contract**: model, parameters, template, eval set, and expected pass rate — code just loads it.
- v1.1.0 renders longer because it adds a few-shot example — a different, still-retrievable version, not an edit of v1.0.0.
- A hardcoded string has **one mutable state and no history**; the registry is the fix.

#### 💡 What just happened

⚡ What just happened? The prompt registry stores each prompt as versioned data with a permanent address (id + semver), loaded by version; the file is the contract, and a published version is never edited in place. Tradeoff: a little loading ceremony, in exchange for full history, diffs, and rollback. Next: what the version number itself should mean - semantic versioning.

- A shelf of versioned prompt cards; pick a version to show its template and eval bar
- A hardcoded-vs-registry toggle showing how much history is retrievable

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 2: Semantic Versioning For Prompts

### Semantic Versioning For Prompts

The version triple encodes the RISK of a change. PATCH: no behaviour change (a typo). MINOR: a backwards-compatible improvement (added examples). MAJOR: a breaking change to the output shape or behaviour - which needs a new eval set and a downstream code change in the same PR. Never edit a published version; log the version on every call.

A version number is only useful if it *means* something. Borrow semantic versioning from software and read the triple as **MAJOR.MINOR.PATCH**, where the bump encodes the **risk** of the change. A **PATCH** is a change with *no behaviour impact* — a typo, whitespace, a clarified comment — and its eval bar is simply “pass rate must not drop”. A **MINOR** is a *backwards-compatible improvement* — adding few-shot examples, tightening an instruction — expected to help or stay flat while keeping the same output shape. A **MAJOR** is a *breaking change*: the output format, the categories, or the behaviour shifts, so downstream code will break unless updated. The rule that saves you: a MAJOR bump ships a **new eval set** and the **downstream code change in the same pull request**, atomically. Two invariants hold across all of them — you **never edit a published version in place** (1.1.0 means one thing forever), and you **log the version on every call**. The block walks a real change history, keyless.

> 💊 **Analogy**
>
> It is **the label on a medicine bottle**. A PATCH is a new label on the same pills — the wording is clearer but the formula is identical, so nothing about the dose changes. A MINOR is an improved formula at the same dose — better, but you take it the same way. A MAJOR is a *different drug*: new active ingredient, new dosing, and you absolutely must re-read the instructions and re-check it against everything else you take. Bumping a prompt is the same triage: the number tells the next engineer how carefully to handle the change and whether the things downstream need to be re-checked.

You add three few-shot examples to a prompt to help it on ambiguous inputs. The output shape is unchanged. What bump?

**📝 `02_semver_for_prompts.py`** - *runnable*

In [ ]:
# Semantic versioning for prompts: the bump number encodes the RISK. PATCH = no behaviour change; MINOR = a
# backwards-compatible improvement; MAJOR = a breaking change (output shape/categories) that needs downstream updates.
RULES = {"patch": "no behaviour change; pass-rate must not drop (typo/whitespace)",
         "minor": "backwards-compatible improvement; pass-rate improves or stays flat",
         "major": "BREAKING output/behaviour change; NEW eval set + update downstream code in the same PR"}
def bump(version, part):
    major, minor, patch = (int(x) for x in version.split("."))
    if part == "major": major, minor, patch = major + 1, 0, 0
    elif part == "minor": minor, patch = minor + 1, 0
    else: patch += 1
    return "{}.{}.{}".format(major, minor, patch)
CHANGES = [("tighten wording 'only' -> 'ONLY'", "patch"),
           ("add 3 few-shot examples for ambiguous reviews", "minor"),
           ("switch plain-text output to JSON {category, confidence}", "major"),
           ("fix a stray trailing space", "patch")]
v = "1.0.0"
print("start        v{}".format(v))
for desc, part in CHANGES:
    v = bump(v, part)
    print("{:<6} -> v{:<7} {:<48} [{}]".format(part.upper(), v, desc, RULES[part].split(";")[0]))
print()
print("The MAJOR bump (v2.0.0) is the dangerous one: {}".format(RULES["major"]))
print("Two hard rules: never edit a published version in place, and log the prompt_version on every call.")

# Output:
# start        v1.0.0
# PATCH  -> v1.0.1   tighten wording 'only' -> 'ONLY'                 [no behaviour change]
# MINOR  -> v1.1.0   add 3 few-shot examples for ambiguous reviews    [backwards-compatible improvement]
# MAJOR  -> v2.0.0   switch plain-text output to JSON {category, confidence} [BREAKING output/behaviour change]
# PATCH  -> v2.0.1   fix a stray trailing space                       [no behaviour change]
#
# The MAJOR bump (v2.0.0) is the dangerous one: BREAKING output/behaviour change; NEW eval set + update downstream code in the same PR
# Two hard rules: never edit a published version in place, and log the prompt_version on every call.

- Each change is classified into a **PATCH, MINOR, or MAJOR** bump, and the version number advances accordingly.
- The history walks 1.0.0 to 1.0.1 (patch) to 1.1.0 (minor) to 2.0.0 (major) to 2.0.1 (patch).
- The **MAJOR** bump — plain text to JSON — is the dangerous one: a new eval set plus a downstream code change in the same PR.
- Two invariants: never edit a published version in place, and log the `prompt_version` on every call.

#### 💡 What just happened

⚡ What just happened? Semantic versioning makes the bump number encode risk: PATCH (no behaviour change), MINOR (compatible improvement), MAJOR (breaking - new eval set + downstream change in the same PR). Tradeoff / rule: never edit a published version, always log the version. Next: how to route traffic between two versions cleanly - deterministic bucketing.

- Pick a change type and watch the version number bump with the right PATCH/MINOR/MAJOR rule
- Walk the version history from 1.0.0 through 2.0.1

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 3: Deterministic Bucketing

### Deterministic Bucketing

To A/B two versions you split users between them - but the split must be DETERMINISTIC. Hash the user id to a stable number in [0,1); the same user always lands in the same arm across requests, servers, and restarts, so no one ever sees both arms and pollutes the metric. Random assignment reshuffles a user every call - the single most common way LLM experiments get contaminated.

To compare two prompt versions you send some users to each — but *how* you assign them decides whether the test means anything. The assignment must be **deterministic**: a given user must land in the *same* arm every single time, on every server, across restarts. The clean way is a **hash**: take the user id (or tenant id), hash it to a stable number in **[0,1)**, and assign the arm by where that number falls. Because the hash is a pure function, you need *no stored table* of who is in which arm — the id alone determines it, forever. The failure this prevents is subtle and common: if you assign with **random** per request, the same person sees the candidate on one call and the champion on the next, so their outcomes are smeared across both arms and the comparison is **polluted**. Consistent bucketing is the single most important thing that keeps an LLM experiment clean. The block routes many users and checks the determinism, keyless.

> 🎟️ **Analogy**
>
> It is **assigning seats by the letters of your name**. Imagine a huge event where your seat is decided by a fixed rule applied to your name — run the rule and you always get the same seat, no guest list required, and you never end up in two sections at once. Deterministic bucketing is that rule: hash the user id and the seat (arm) falls out, the same way every time. Assigning seats by a coin flip at the door instead — that is `random` — would put the same guest in a different section every time they stepped out and came back, which is exactly how an A/B test gets contaminated.

You route A/B traffic with random per request instead of a hash of the user id. What goes wrong?

**📝 `03_deterministic_bucketing.py`** - *runnable*

In [ ]:
# A/B routing needs DETERMINISTIC bucketing: hash the user id to a stable number in [0,1). The SAME user always
# lands in the SAME arm - across requests, servers, and restarts - so no user ever sees both arms and pollutes the test.
import hashlib
def bucket(user_id, experiment="review-classifier-ab"):
    h = int(hashlib.md5((experiment + "::" + user_id).encode()).hexdigest(), 16)
    return (h % 10_000) / 10_000            # stable, uniform-ish in [0.0, 1.0)
CANDIDATE_SHARE = 0.10                        # send 10% to the candidate
def arm(user_id):
    return "candidate" if bucket(user_id) < CANDIDATE_SHARE else "champion"
users = ["u{:05d}".format(i) for i in range(2000)]
cand = sum(1 for u in users if arm(u) == "candidate")
print("Routing {} users with a {:.0%} candidate share:".format(len(users), CANDIDATE_SHARE))
print("  candidate arm: {} users ({:.1%})   champion arm: {} users ({:.1%})".format(cand, cand / len(users), len(users) - cand, 1 - cand / len(users)))
print("  the split lands near the {:.0%} target from the hash alone - no stored user->arm table needed.".format(CANDIDATE_SHARE))
print()
print("Determinism check: user 'u00042' -> arm '{}' on every call (bucket {:.4f}).".format(arm("u00042"), bucket("u00042")))
print("  ...checked twice, same result: '{}' and '{}'.".format(arm("u00042"), arm("u00042")))
print("random.random() would reassign that user every request, so one person would see BOTH arms - test polluted.")

# Output:
# Routing 2000 users with a 10% candidate share:
#   candidate arm: 208 users (10.4%)   champion arm: 1792 users (89.6%)
#   the split lands near the 10% target from the hash alone - no stored user->arm table needed.
#
# Determinism check: user 'u00042' -> arm 'champion' on every call (bucket 0.5015).
#   ...checked twice, same result: 'champion' and 'champion'.
# random.random() would reassign that user every request, so one person would see BOTH arms - test polluted.

- Hashing the user id splits the users near the target share — here a candidate share of about one in ten — from the **hash alone**, no stored table.
- A specific user maps to a **fixed bucket** and therefore a fixed arm — checked twice, same result.
- That determinism is what keeps each user in **one arm only**, so their outcomes are not smeared across both.
- `random` would reassign the user every request — the classic way an A/B test gets polluted.

#### 💡 What just happened

⚡ What just happened? Deterministic bucketing hashes the user id to a stable arm, so the same user always sees the same version and no one pollutes the test by seeing both; it needs no stored assignment table. Tradeoff: none worth the name - it is strictly better than random for experiments. Next: running the A/B test and reading each arm.

- Two thousand user dots fall through a hash into two arms at a share you set
- A checked user stays in the same arm until a random toggle makes it flicker

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 4: The A/B Test Itself

### The A/B Test Itself

Route each user by the deterministic bucket, serve that arm’s prompt, and log the result with its prompt_version. Grouping the trace log by prompt_version gives each arm’s pass rate. Because output is non-deterministic prose, the pass/fail is a judge’s verdict, not an exact-match - which is why an A/B needs the evals of the next lessons.

Now run the test. Each request is routed by the user’s deterministic bucket, served that arm’s prompt version, and **logged with its `prompt_version`** — the trace field from 14.1. Afterwards you **group the log by `prompt_version`** and compute each arm’s pass rate, cost, and latency. There is one twist unique to LLMs: the “pass” is not an exact-match against a label, because the output is open-ended prose. It is a **judge’s verdict** — an LLM-as-judge or a rubric score — which is precisely why A/B testing prompts *depends on* the evaluation machinery you build in the next lessons. In the block the verdict is a deterministic stand-in so it runs keyless, but the shape is real: route, serve, log, group. The block tallies both arms, keyless.

> 🍴 **Analogy**
>
> It is **two kitchens cooking from different recipes**. You send some diners to kitchen A and some to kitchen B, each cooking the same dish from a slightly different recipe, and then you compare the reviews. The crucial detail is that a given diner always eats at the *same* kitchen (that is the deterministic bucketing), and every plate is tagged with which kitchen made it (that is logging the `prompt_version`), so when you read the reviews you can honestly say which recipe scored better. Without the tags you would have a pile of reviews and no idea which kitchen earned them.

Your A/B log has a prompt_version on every call. What is the right way to get each arm’s pass rate?

**📝 `04_ab_test.py`** - *runnable*

In [ ]:
# The A/B test itself: route each user by the deterministic bucket, serve that arm's prompt, log the result per
# version. Grouping by prompt_version gives the per-arm pass rate. (Pass/fail here is a deterministic stand-in for a judge.)
import hashlib
def bucket(user_id, experiment="review-classifier-ab"):
    return (int(hashlib.md5((experiment + "::" + user_id).encode()).hexdigest(), 16) % 10_000) / 10_000
def arm(user_id):
    return "candidate" if bucket(user_id) < 0.10 else "champion"
TRUE_RATE = {"champion": 0.92, "candidate": 0.95}     # the (unknown-in-real-life) quality of each prompt
def judged_pass(user_id, rate):                        # deterministic stand-in for an LLM-as-judge verdict
    h = int(hashlib.md5(("verdict::" + user_id).encode()).hexdigest(), 16)
    return (h % 10_000) / 10_000 < rate
users = ["u{:05d}".format(i) for i in range(2000)]
tally = {"champion": [0, 0], "candidate": [0, 0]}      # [passes, total]
for u in users:
    a = arm(u); tally[a][1] += 1
    if judged_pass(u, TRUE_RATE[a]): tally[a][0] += 1
print("Per-arm results (group the trace log by prompt_version):")
for a in ("champion", "candidate"):
    passes, total = tally[a]
    print("  {:<9} n={:<5} passes={:<5} observed pass-rate={:.1%}".format(a, total, passes, passes / total))
print()
print("The candidate looks better here, but the candidate arm only saw {} requests.".format(tally["candidate"][1]))
print("Is the gap real, or noise? A point estimate is not enough - you test it for significance next.")

# Output:
# Per-arm results (group the trace log by prompt_version):
#   champion  n=1792  passes=1653  observed pass-rate=92.2%
#   candidate n=208   passes=196   observed pass-rate=94.2%
#
# The candidate looks better here, but the candidate arm only saw 208 requests.
# Is the gap real, or noise? A point estimate is not enough - you test it for significance next.

- Each user is routed by bucket, served that arm’s prompt, and the result is logged with its `prompt_version`.
- Grouping by version gives each arm’s pass rate — the champion and the candidate, side by side.
- The candidate looks a little better, but its arm saw far fewer requests — a small sample.
- The “pass” here is a deterministic stand-in for a judge’s verdict — the real one comes from the evals in Lesson 14.3.

#### 💡 What just happened

⚡ What just happened? The A/B test routes by bucket, serves each arm’s prompt, logs the prompt_version, and groups the log to get each arm’s pass rate; the pass is a judge’s verdict, so A/B needs evals. Tradeoff: the winner is only as trustworthy as the judge and the sample. Next: is the gap real, or noise - the significance test.

- Two prompt arms whose per-arm pass-rate bars fill as traffic streams
- n counters show the champion arm dwarfs the candidate arm

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 5: Statistical Significance

### Statistical Significance

Do not ship on a point estimate. A two-proportion z-test asks whether the observed gap could be noise. The SAME lift is not significant on a small sample but is on a large one - sample size, not the gap, decides. Small lifts need huge samples, so low-traffic features run longer, use a richer metric, or ship only large effects. (Callback: Lesson 1.8 Bayesian A/B.)

Here is where most teams go wrong: they see the candidate a few points ahead and ship it. But a lead on a small sample is usually **noise**. The frequentist tool is a **two-proportion z-test**, which answers one question — *could a gap this size have appeared by chance if the two prompts were actually equal?* If that probability (the p-value) is small, the difference is real; if not, you cannot distinguish it from luck. The counter-intuitive part is that the **same** lift can be firmly significant or completely inconclusive depending only on the **sample size**: a few-point gap over a hundred requests per arm is noise, while the identical gap over ten thousand per arm is a clear win. And the flip side sets your expectations — small lifts need *large* samples, so a low-traffic feature must either run for a long time, use a richer metric than pass/fail, or adopt a “large effect or no ship” rule. This is the frequentist companion to the Bayesian A/B testing you met in Lesson 1.8. The block runs the test at two sample sizes, keyless.

> 🪙 **Analogy**
>
> It is **refusing to call a coin biased after three flips**. If you flip a coin three times and get two heads, you have learned essentially nothing — that happens constantly with a fair coin. You would need *hundreds* of flips before a small lean toward heads became convincing. An A/B test is the same: a few-point edge over a handful of requests is the two-heads-in-three of prompt testing. The significance test is just the discipline of asking “have I flipped enough times to trust this lean?” before you bet the product on it.

The candidate is three points ahead of the champion. When is that gap trustworthy?

**📝 `05_significance.py`** - *runnable*

In [ ]:
# Do NOT ship on a point estimate. A two-proportion z-test asks: could this gap be noise? Same +3pp lift is
# NOT significant at n=100 per arm but IS at n=10000 per arm. Sample size, not the gap, decides. (Callback: Lesson 1.8.)
import math
def two_prop_p(p1, n1, p2, n2):
    x1, x2 = p1 * n1, p2 * n2
    pool = (x1 + x2) / (n1 + n2)
    se = math.sqrt(pool * (1 - pool) * (1 / n1 + 1 / n2))
    if se == 0: return 0.0, 999.0
    z = (p2 - p1) / se
    p_two_sided = 2 * (1 - 0.5 * (1 + math.erf(abs(z) / math.sqrt(2))))   # normal CDF via erf, no scipy
    return z, p_two_sided
for label, n in [("small test", 100), ("large test", 10_000)]:
    z, p = two_prop_p(0.92, n, 0.95, n)
    verdict = "SHIP (significant)" if p < 0.05 else "WAIT (could be noise)"
    print("{:<11} 92% vs 95% (+3pp), n={:>6}/arm -> z={:>5.2f}  p={:.4f}  -> {}".format(label, n, z, p, verdict))
print()
# how many samples per arm to detect a given lift around a 90% baseline (approx two-proportion sizing, alpha .05 / power .8)
def sample_size(p, lift, z_a=1.96, z_b=0.84):
    p2 = p + lift; pbar = (p + p2) / 2
    return math.ceil((z_a * math.sqrt(2 * pbar * (1 - pbar)) + z_b * math.sqrt(p * (1 - p) + p2 * (1 - p2))) ** 2 / lift ** 2)
for lift in (0.01, 0.03, 0.05):
    print("to detect a {:>2.0f}pp lift around a 90% baseline you need about {:>6} requests PER ARM.".format(lift * 100, sample_size(0.90, lift)))
print("Small lifts need huge samples. Low-traffic features: run longer, use a richer metric, or ship only large effects.")

# Output:
# small test  92% vs 95% (+3pp), n=   100/arm -> z= 0.86  p=0.3895  -> WAIT (could be noise)
# large test  92% vs 95% (+3pp), n= 10000/arm -> z= 8.60  p=0.0000  -> SHIP (significant)
#
# to detect a  1pp lift around a 90% baseline you need about  13480 requests PER ARM.
# to detect a  3pp lift around a 90% baseline you need about   1354 requests PER ARM.
# to detect a  5pp lift around a 90% baseline you need about    434 requests PER ARM.
# Small lifts need huge samples. Low-traffic features: run longer, use a richer metric, or ship only large effects.

- The **same lift** is tested at two sample sizes: on the small sample the p-value is large (could be noise, WAIT), on the large sample it is tiny (significant, SHIP).
- So **sample size, not the point estimate**, decides whether a gap is real.
- The sizing table shows small lifts need **large samples** — a one-point lift near a high baseline needs many thousands of requests per arm.
- Low-traffic features: run longer, use a richer metric than pass/fail, or ship only large effects.

#### 💡 What just happened

⚡ What just happened? A two-proportion z-test asks whether the gap could be noise; the same lift is inconclusive on a small sample and significant on a large one, so sample size decides. Tradeoff: small lifts need big samples, so low-traffic features must run longer or demand large effects. Next: even a real winner ramps gradually - the canary.

- A sample-size slider drives a p-value bar under the 0.05 line so a WAIT flips to SHIP
- A sample-size table for one-, three-, and five-point lifts

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 6: Canary Rollout

### Canary Rollout

A winning A/B does not jump to 100 percent. Canary ramps 5 to 25 to 50 to 100, and each stage has TIGHTER rollback triggers than the last, because more users mean less tolerance for regression. A trigger trip auto-rolls-back. The modern order is shadow (score both, serve neither) first, then canary, then ramp - so failures surface before any user is exposed.

Even after an A/B test declares a winner, you do not flip it to everyone at once. A **canary rollout** ramps the new version through stages — a small slice, then a quarter, then half, then all — watching guardrail metrics at each step, with **automatic rollback** if a trigger trips. The key discipline is that the triggers get **tighter as you ramp**: a small pass-rate dip you would tolerate at a five-percent slice becomes a rollback trigger at half, because more users are exposed and backing out is more expensive. This staged approach catches the failures a sample cannot: format breakages that only appear on rare inputs, tail-latency regressions, cost creep, and tone shifts that draw complaints. The 2026 refinement adds a step *before* the canary: **shadow testing** runs the new prompt on live inputs but *serves neither* its output to users — it just scores both — so you catch problems at **zero user risk** before exposing anyone. The order is **shadow, then canary, then ramp**. The block runs a staged canary that rolls back, keyless.

> 🥚 **Analogy**
>
> It is **a taste-tester before the banquet**. Before the whole hall is served, one taster tries the dish; if it is fine, a table tries it; then a section; then everyone. And the standard *rises* as you go: a faint off-note the taster might wave through becomes an immediate stop-the-service once half the room has plates in front of them, because the cost of a bad dish grows with how many people are eating it. A canary rollout is that escalating caution — and shadow testing is the chef quietly tasting a spoonful in the kitchen before a single guest is served at all.

Your canary’s rollback trigger for a pass-rate drop is looser at the five-percent stage than at the fifty-percent stage. Why?

**📝 `06_canary_rollout.py`** - *runnable*

In [ ]:
# A winning A/B does NOT go straight to 100%. Canary ramps 5 -> 25 -> 50 -> 100, and each stage has TIGHTER
# rollback triggers than the last (more users = less tolerance). A trigger trip auto-rolls-back. Shadow-test first for zero user risk.
STAGES = [(5, {"pass_drop": 2.0, "cost": 30, "p99": 2.0}),   # (traffic %, rollback-if thresholds)
          (25, {"pass_drop": 1.0, "cost": 15, "p99": 1.5}),
          (50, {"pass_drop": 0.5, "cost": 10, "p99": 1.5}),
          (100, None)]
OBSERVED = {5:  {"pass_drop": 0.1, "cost": 6,  "p99": 1.1},   # measured metrics at each stage (illustrative)
            25: {"pass_drop": 0.3, "cost": 9,  "p99": 1.2},
            50: {"pass_drop": 0.8, "cost": 9,  "p99": 1.3}}    # pass_drop 0.8 trips the tight 0.5 trigger at 50%
for pct, trig in STAGES:
    if trig is None:
        print("stage {:>3}%  -> FULL ROLLOUT reached".format(pct)); break
    m = OBSERVED[pct]
    tripped = [k for k in trig if m[k] > trig[k]]
    if tripped:
        print("stage {:>3}%  triggers {}  observed {}  -> ROLLBACK (tripped: {})".format(pct, trig, m, ", ".join(tripped))); break
    print("stage {:>3}%  observed pass_drop={}pp cost=+{}% p99={}x  -> all triggers pass, ADVANCE".format(pct, m["pass_drop"], m["cost"], m["p99"]))
print()
print("The 50% stage rolls back: a 0.8pp pass drop is fine at 5% (trigger 2.0) but trips the tighter 0.5 trigger at 50%.")
print("Modern order (2026): shadow (score both, serve neither) -> canary (1-5%, eval-gated) -> ramp. Rollback is automatic.")

# Output:
# stage   5%  observed pass_drop=0.1pp cost=+6% p99=1.1x  -> all triggers pass, ADVANCE
# stage  25%  observed pass_drop=0.3pp cost=+9% p99=1.2x  -> all triggers pass, ADVANCE
# stage  50%  triggers {'pass_drop': 0.5, 'cost': 10, 'p99': 1.5}  observed {'pass_drop': 0.8, 'cost': 9, 'p99': 1.3}  -> ROLLBACK (tripped: pass_drop)
#
# The 50% stage rolls back: a 0.8pp pass drop is fine at 5% (trigger 2.0) but trips the tighter 0.5 trigger at 50%.
# Modern order (2026): shadow (score both, serve neither) -> canary (1-5%, eval-gated) -> ramp. Rollback is automatic.

- The rollout ramps through stages, checking the observed metrics against that stage’s **rollback triggers**.
- The triggers **tighten** at each stage — the same pass-rate dip that passes early trips a rollback later.
- This run **rolls back at the half stage**: a dip that was fine at the small slice trips the tighter trigger.
- The 2026 order is **shadow** (score both, serve neither) then **canary** then **ramp** — failures surface before any user is exposed.

#### 💡 What just happened

⚡ What just happened? A canary ramps a winner through stages with tightening rollback triggers (more users, less tolerance) and auto-rolls-back on a trip; shadow-testing first catches failures at zero user risk. Tradeoff: a slower rollout, in exchange for catching load-only and edge-case failures. Next: what a rollback does - and does not - undo.

- A staged ramp with tightening rollback triggers; an observed-metric stream trips one stage into rollback
- The 2026 order: shadow, then canary, then ramp

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 7: The Rollback Path

### The Rollback Path

Reverting the served version is fast - seconds to minutes by mechanism. But it does NOT undo the side effects the bad version already produced: responses cached from it, database rows it wrote, and requests already in flight. The fix is not faster reverts; it is TAGGING every cache entry and every database write with the prompt_version that made it, so a rollback can find and heal them.

You will roll back — plan for it as a feature, not an emergency. The easy half is reverting the **served version**, and its speed depends on the mechanism: a YAML-in-git registry needs a redeploy (minutes), a database flag flips in seconds, a platform rolls back in one click. But here is the part that surprises everyone: **reverting the prompt does not undo what the bad version already did**. Three footprints persist. **Cached responses** from the bad version keep serving its output until their TTL expires. **Database rows it wrote** — a label, a summary, a classification — stay in your data, produced by a version you have since retracted. And **requests already in flight** have shipped; you cannot recall them. So the fastest possible revert still leaves a footprint in your caches and your data. The fix is not speed — it is **tagging**: stamp every cache key and every database write with the `prompt_version` that produced it, so a rollback can *find* what the bad version touched and heal it. The block models the revert times and the footprint, keyless.

> ✉️ **Analogy**
>
> It is the difference between **undo on a document and unsend on an email**. Undo instantly restores the document to how it was — that is reverting the served prompt version. But if you already emailed the document to a thousand people, undo does nothing about the copies in their inboxes — those are the cached responses, the rows written to your database, and the requests already in flight. You cannot unsend them. The only real remedy is to have labelled every copy with which draft it came from, so you can go find the ones from the bad draft and correct them. That label is the `prompt_version` on every write.

You flip the served prompt back to the previous version in ten seconds. Is the bad version’s impact gone?

**📝 `07_rollback.py`** - *runnable*

In [ ]:
# Rollback speed is the EASY part. The hard part: reverting the prompt does NOT undo the side effects v2 already
# produced. Tag every cache entry and DB write with the prompt_version that made it, or you cannot find and heal them.
MECHANISMS = [("YAML in git, redeploy", 600), ("DB flag / active_version column", 30), ("prompt platform, one click", 10)]
print("Time to revert the SERVED version:")
for name, secs in MECHANISMS:
    print("  {:<34} ~{:>4} s".format(name, secs))
print()
AFTER_ROLLBACK = [("the served prompt version", "reverts immediately"),
                  ("responses cached from v2", "keep serving v2 output until the TTL expires"),
                  ("rows v2 wrote to your database", "stay v2-labelled forever unless you reprocess them"),
                  ("requests already in flight", "already shipped - you cannot recall them")]
print("What a rollback does NOT undo:")
for what, fate in AFTER_ROLLBACK:
    print("  - {:<32} {}".format(what, fate))
print()
print("So the fastest revert still leaves a v2 footprint in caches and data. The fix is not speed, it is TAGGING:")
print("stamp every cache key and every DB write with prompt_version, so a rollback can find and heal what v2 touched.")

# Output:
# Time to revert the SERVED version:
#   YAML in git, redeploy              ~ 600 s
#   DB flag / active_version column    ~  30 s
#   prompt platform, one click         ~  10 s
#
# What a rollback does NOT undo:
#   - the served prompt version        reverts immediately
#   - responses cached from v2         keep serving v2 output until the TTL expires
#   - rows v2 wrote to your database   stay v2-labelled forever unless you reprocess them
#   - requests already in flight       already shipped - you cannot recall them
#
# So the fastest revert still leaves a v2 footprint in caches and data. The fix is not speed, it is TAGGING:
# stamp every cache key and every DB write with prompt_version, so a rollback can find and heal what v2 touched.

- Reverting the **served version** is fast, and the time depends on the mechanism — a redeploy, a database flag, or a platform click.
- But three footprints **persist**: cached responses (until TTL), database rows the bad version wrote, and in-flight requests.
- So the fastest revert still leaves a **v2 footprint** in your caches and data.
- The fix is **tagging** — stamp every cache entry and database write with the `prompt_version` so a rollback can find and heal them.

#### 💡 What just happened

⚡ What just happened? Reverting the served version is fast, but it does not undo cached responses, database rows the version wrote, or in-flight requests; the fix is tagging every write with prompt_version so a rollback can find and heal the footprint. That is the whole lifecycle: register it, version it, route it, test it, ramp it, and be able to undo it.

- A revert control flips the served version fast while a cache-TTL bar, database rows, and in-flight requests persist
- A tag-everything toggle makes the footprint findable and healable

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

## Putting It Together

> ✅ **Info**
>
> 🧠 The whole picture
> The prompt lifecycle is one continuous discipline. It starts with a **registry** that gives every version a permanent address (Step 1), numbered by **semantic versioning** so the bump encodes the risk (Step 2). To improve it safely you route traffic with **deterministic bucketing** so each user stays in one arm (Step 3), run the **A/B test** by grouping the trace log per version (Step 4), and only trust the winner when the gap clears a **significance test** (Step 5). Then you ramp it through a **canary** with tightening triggers, shadow-testing first (Step 6), and keep a **rollback path** that tags every write so you can heal the footprint (Step 7). Ask two questions of any prompt change: does a version number and an eval stand behind it, and can you both prove it is better and undo it if it is not?

**📦 **The prompt-versioning decision map****

| Where you are | Storage & versioning | A/B | Tool (2026) |
|---|---|---|---|
| Hackathon / demo | hardcoded; git on the file | none | — |
| Pilot (a few prompts) | YAML in git, SemVer in the filename | hash-based bucketing (this lesson) | custom (~80 lines) |
| Internal tool (many prompts) | YAML or a database registry | multi-arm bucketing | custom, or Pezzo / Langfuse |
| Customer-facing, multi-team | platform-backed + audit log | native A/B + canary | LangSmith / PromptLayer / Braintrust |
| Regulated / EU residency | self-hosted platform + approvals | strict canary + human sign-off | Langfuse (self-hosted) |

> 📦 **Info**
>
> ➡️ Where this goes nextEvery A/B test in this lesson ended on a “pass” that a judge has to define. Building that judge — golden sets, LLM-as-judge, calibration, and the CI eval gate that decides a winner — is the work of Lesson 14.3. Turning the losing arm’s failures into a taxonomy and custom evals is error analysis in Lesson 14.4. And serving the winning prompt cheaply, with caching and routing, is Lesson 14.5. The primary references are the [SemVer specification](https://semver.org/), the [Langfuse prompt-management docs](https://langfuse.com/docs/prompt-management), and the [Jinja templating docs](https://jinja.palletsprojects.com/).

*Practice exercises are in the companion practice notebook.*

---

## 🎓 Summary

You've completed the practical part of **Prompt Versioning& A/B Testing**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `practice-lab/practice-lab-14_2.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-14.2.html` - regenerate this notebook whenever the source HTML is updated.*
